# Chapter 3: Coding Attention Mechanisms

Companion notebook for [Chapter 3 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/attention). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## Setup

Code carried over from earlier chapters, plus anything this chapter needs to run. Run it once and move on.

In [ ]:
#@title Setup (run me first)

import math
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- from Chapter 2 ----
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class NextTokenDataset(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        self.inputs, self.targets = [], []
        for start in range(0, len(ids) - context_length, stride):
            window = ids[start : start + context_length + 1]
            self.inputs.append(torch.tensor(window[:-1]))   # all but the last
            self.targets.append(torch.tensor(window[1:]))   # all but the first

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]


def create_dataloader(text, batch_size=8, max_length=256, stride=256,
                      shuffle=True, drop_last=True):
    dataset = NextTokenDataset(text, tiktoken.get_encoding("gpt2"),
                               context_length=max_length, stride=stride)
    return DataLoader(dataset, batch_size=batch_size,
                      shuffle=shuffle, drop_last=drop_last)

## Why attention?

## A simple self-attention, no weights

In [ ]:
import torch

inputs = torch.tensor([
    [0.2, 0.7, 0.1],  # My
    [0.9, 0.3, 0.2],  # cat
    [0.7, 0.2, 0.5],  # naps
    [0.1, 0.6, 0.3],  # in
    [0.3, 0.1, 0.9],  # warm
    [0.4, 0.2, 0.8],  # sun
])

### Step 1: attention scores

In [ ]:
query = inputs[2]                 # "naps"
scores = inputs @ query           # one dot product per token
print(scores)
# tensor([0.3300, 0.7900, 0.7800, 0.3400, 0.6800, 0.7200])

### Step 2: normalise with softmax

In [ ]:
weights = torch.softmax(scores, dim=0)
print(weights.sum())   # tensor(1.)

### Step 3: the context vector

In [ ]:
context = weights @ inputs        # weighted sum of all embeddings

### All tokens at once

In [ ]:
scores = inputs @ inputs.T                 # (6, 6): row i = query i vs all tokens
weights = torch.softmax(scores, dim=-1)    # softmax along each row
contexts = weights @ inputs                # (6, 3): one context vector per token

## Self-attention with trainable weights

In [ ]:
torch.manual_seed(0)
d_in, d_out = 3, 2

W_q = torch.rand(d_in, d_out)
W_k = torch.rand(d_in, d_out)
W_v = torch.rand(d_in, d_out)

Q = inputs @ W_q     # (6, 2)
K = inputs @ W_k     # (6, 2)
V = inputs @ W_v     # (6, 2)

scores = Q @ K.T     # (6, 6)

### Why divide by √d_k?

In [ ]:
d_k = K.shape[-1]
weights = torch.softmax(scores / d_k**0.5, dim=-1)
contexts = weights @ V        # (6, 2)

### A compact PyTorch module

In [ ]:
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.to_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_v = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):                  # x: (num_tokens, d_in)
        q, k, v = self.to_q(x), self.to_k(x), self.to_v(x)
        scores = q @ k.T
        weights = torch.softmax(scores / k.shape[-1] ** 0.5, dim=-1)
        return weights @ v                 # (num_tokens, d_out)

torch.manual_seed(123)
attn = SelfAttention(d_in=3, d_out=2)
print(attn(inputs).shape)   # torch.Size([6, 2])

## Causal attention

### Masking with −∞

In [ ]:
n = scores.shape[0]
future = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)

masked = scores.masked_fill(future, float('-inf'))
weights = torch.softmax(masked / d_k**0.5, dim=-1)

### Dropout on attention weights

In [ ]:
dropout = nn.Dropout(p=0.5)
print(dropout(torch.ones(6)))
# e.g. tensor([2., 0., 0., 2., 2., 0.])

### A causal attention module

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.to_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)

    def forward(self, x):                   # x: (batch, n, d_in)
        n = x.shape[1]
        q, k, v = self.to_q(x), self.to_k(x), self.to_v(x)
        scores = q @ k.transpose(1, 2)      # (batch, n, n)
        scores = scores.masked_fill(self.mask[:n, :n], float('-inf'))
        weights = torch.softmax(scores / k.shape[-1] ** 0.5, dim=-1)
        weights = self.dropout(weights)
        return weights @ v                  # (batch, n, d_out)

batch = torch.stack([inputs, inputs])       # (2, 6, 3)
ca = CausalAttention(d_in=3, d_out=2, context_length=6, dropout=0.0)
print(ca(batch).shape)   # torch.Size([2, 6, 2])

## Multi-head attention

### The simple way: stack heads

In [ ]:
class MultiHeadWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
            for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

mha = MultiHeadWrapper(3, 2, context_length=6, dropout=0.0, num_heads=2)
print(mha(batch).shape)   # torch.Size([2, 6, 4])  (2 heads × 2 dims)

### The efficient way: split one big projection

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.to_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)

    def split_heads(self, t):
        b, n, _ = t.shape
        # (b, n, d_out) -> (b, n, heads, head_dim) -> (b, heads, n, head_dim)
        return t.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        b, n, _ = x.shape
        q = self.split_heads(self.to_q(x))
        k = self.split_heads(self.to_k(x))
        v = self.split_heads(self.to_v(x))

        scores = q @ k.transpose(2, 3)                    # (b, heads, n, n)
        scores = scores.masked_fill(self.mask[:n, :n], float('-inf'))
        weights = torch.softmax(scores / self.head_dim ** 0.5, dim=-1)
        weights = self.dropout(weights)

        out = weights @ v                                 # (b, heads, n, head_dim)
        out = out.transpose(1, 2).contiguous().view(b, n, -1)  # concat heads
        return self.out_proj(out)                         # (b, n, d_out)

In [ ]:
torch.manual_seed(123)
mha = MultiHeadAttention(d_in=3, d_out=4, context_length=6, dropout=0.0, num_heads=2)
print(mha(batch).shape)   # torch.Size([2, 6, 4])

## Summary